# Chapter 5 — reproducible result tables and figures

This notebook is the presentation layer for Chapter 5. It reads only saved
`unified_eval_v4` and causal-suite artifacts, validates the canonical bf16
grid, writes compact CSV/JSON tables under `Main_Experimentation_Results`, and
creates the Greek-language figures inserted in the thesis.

The main and multi-seed outputs are generated independently of the causal
section. The causal section is deliberately fail-closed: it reports missing or
stale cells and does not create a figure until every required condition uses
the canonical checkpoint and, for JEPA, the validation-selected `best`
readout.


In [1]:
from pathlib import Path
from itertools import product
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

PROJECT_CANDIDATES = [
    Path(r"C:/Users/Babis/Dropbox/MSc_AI/Master_Thesis_Code_Final"),
    Path("/content/drive/Othercomputers/MyLaptop/Master_Thesis_Code_Final"),
    Path("/content/drive/MyDrive/Master_Thesis_Code_Final"),
]
ARTIFACT_CANDIDATES = [
    Path(r"G:/My Drive/Master_Thesis_Artifacts"),
    Path("/content/drive/MyDrive/Master_Thesis_Artifacts"),
]

PROJECT_ROOT = next(path for path in PROJECT_CANDIDATES if path.is_dir())
ARTIFACTS_ROOT = next(path for path in ARTIFACT_CANDIDATES if path.is_dir())
RESULTS_ROOT = PROJECT_ROOT / "Main_Experimentation_Results"
OUTPUT_DIR = RESULTS_ROOT / "Chapter_5"
FIGURE_DIR = PROJECT_ROOT / "docs" / "thesis" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

PROTOCOL = "unified_eval_v4"
BOARDS = (8, 12, 16)
SYSTEMS = (
    ("Transformer-AR", "transformer", "ar"),
    ("Transformer-JEPA", "transformer", "jepa"),
    ("Mamba-AR", "mamba", "ar"),
    ("Mamba-JEPA", "mamba", "jepa"),
)
RUNS = {
    ("Transformer-AR", 8): "Transformer-AR/transformer_ar_b8_bf16",
    ("Transformer-AR", 12): "Transformer-AR/transformer_ar_b12",
    ("Transformer-AR", 16): "Transformer-AR/transformer_ar_b16",
    ("Transformer-JEPA", 8): "Transformer-JEPA/jepa_v5_infonce_b8_run_001_hd_allpos_bf16",
    ("Transformer-JEPA", 12): "Transformer-JEPA/transformer_jepa_v5_hd_infonce_allpos_b12",
    ("Transformer-JEPA", 16): "Transformer-JEPA/transformer_jepa_v5_hd_infonce_allpos_b16",
    ("Mamba-AR", 8): "Mamba-AR/mamba_ar_8x8_bf16",
    ("Mamba-AR", 12): "Mamba-AR/mamba_ar_b12",
    ("Mamba-AR", 16): "Mamba-AR/mamba_ar_b16",
    ("Mamba-JEPA", 8): "Mamba-JEPA/mamba_jepa_v5_hd_infonce_allpos_b8",
    ("Mamba-JEPA", 12): "Mamba-JEPA/mamba_jepa_v5_hd_infonce_allpos_b12",
    ("Mamba-JEPA", 16): "Mamba-JEPA/mamba_jepa_v5_hd_infonce_allpos_b16",
}

SURFACE, INK, INK2, GRID = "#fcfcfb", "#111111", "#55534f", "#deddd9"
COLORS = {
    "Transformer-AR": "#6F9FC8",
    "Transformer-JEPA": "#79B7A3",
    "Mamba-AR": "#D6A24B",
    "Mamba-JEPA": "#C97979",
}
MARKERS = {
    "Transformer-AR": "o",
    "Transformer-JEPA": "o",
    "Mamba-AR": "s",
    "Mamba-JEPA": "s",
}

def load_json(path):
    with Path(path).open(encoding="utf-8") as handle:
        return json.load(handle)

def main_result_path(system, board):
    return ARTIFACTS_ROOT / "runs" / RUNS[(system, board)] / "thesis_eval" / "final" / f"results__{PROTOCOL}.json"

def choose_legal_readout(payload, objective):
    if objective == "ar":
        return "native", payload["native_ar"]
    frozen = payload["frozen_next_move"]
    scores = {
        head: float(frozen[head]["train"]["best_selection_legal_probability_mass"])
        for head in ("linear", "mlp")
    }
    selected = max(("linear", "mlp"), key=lambda head: (scores[head], head == "linear"))
    return selected, frozen[selected]["test_legal"]

print("project  :", PROJECT_ROOT)
print("artifacts:", ARTIFACTS_ROOT)
print("output   :", OUTPUT_DIR)


project  : C:\Users\Babis\Dropbox\MSc_AI\Master_Thesis_Code_Final
artifacts: G:\My Drive\Master_Thesis_Artifacts
output   : C:\Users\Babis\Dropbox\MSc_AI\Master_Thesis_Code_Final\Main_Experimentation_Results\Chapter_5


## 1. Canonical 2×2×3 grid

The next cell validates all twelve seed-42 runs, selects each JEPA next-move
readout only from its saved validation score, and exports both headline and
per-layer board-state metrics.


In [2]:
main_rows, layer_rows, compact_payload = [], [], {"protocol": PROTOCOL, "runs": []}
for system, architecture, objective in SYSTEMS:
    for board in BOARDS:
        path = main_result_path(system, board)
        if not path.is_file():
            raise FileNotFoundError(path)
        payload = load_json(path)
        metadata = payload["metadata"]
        precision = str(metadata.get("training_precision") or metadata.get("train_config", {}).get("precision", "")).lower()
        if precision != "bf16":
            raise RuntimeError(f"{system} {board}x{board}: expected bf16, found {precision!r}")
        if int(metadata["board_size"]) != board:
            raise RuntimeError(f"Board mismatch in {path}")

        readout, legal = choose_legal_readout(payload, objective)
        linear = payload["board_state"]["linear"]["selected"]
        mlp = payload["board_state"]["mlp"]["selected"]
        row = {
            "system": system,
            "architecture": architecture,
            "objective": objective,
            "board": board,
            "seed": int(metadata["train_config"]["seed"]),
            "precision": precision,
            "legal_readout": readout,
            "top1_legal": float(legal["top1_legal_per_game"]),
            "legal_probability_mass": float(legal["legal_prob_mass_per_game"]),
            "top1_ci_low": float(legal["bootstrap_ci_95_per_game"]["top1_legal"]["low"]),
            "top1_ci_high": float(legal["bootstrap_ci_95_per_game"]["top1_legal"]["high"]),
            "legal_mass_ci_low": float(legal["bootstrap_ci_95_per_game"]["legal_probability_mass"]["low"]),
            "legal_mass_ci_high": float(legal["bootstrap_ci_95_per_game"]["legal_probability_mass"]["high"]),
            "linear_absolute_macro": float(linear["absolute"]["test"]["macro_balanced_accuracy"]),
            "linear_absolute_overall": float(linear["absolute"]["test"]["accuracy"]),
            "linear_relative_macro": float(linear["relative"]["test"]["macro_balanced_accuracy"]),
            "linear_relative_overall": float(linear["relative"]["test"]["accuracy"]),
            "linear_relative_layer": int(linear["relative"]["layer"]),
            "mlp_absolute_macro": float(mlp["absolute"]["test"]["macro_balanced_accuracy"]),
            "mlp_absolute_overall": float(mlp["absolute"]["test"]["accuracy"]),
            "mlp_relative_macro": float(mlp["relative"]["test"]["macro_balanced_accuracy"]),
            "mlp_relative_overall": float(mlp["relative"]["test"]["accuracy"]),
            "mlp_relative_layer": int(mlp["relative"]["layer"]),
            "checkpoint_sha256": metadata["checkpoint_sha256"],
            "split_manifest_hash": metadata["split_manifest_hash"],
            "source_results": str(path),
        }
        main_rows.append(row)
        compact_payload["runs"].append(row)

        probe = payload["board_state"]["linear"]
        layer_count = max(int(layer) for layer in probe["test_metrics"])
        selected_layer = int(probe["selected"]["relative"]["layer"])
        for layer_text, metrics in probe["test_metrics"].items():
            layer = int(layer_text)
            layer_rows.append({
                "system": system,
                "board": board,
                "layer": layer,
                "normalized_depth": layer / layer_count,
                "relative_macro": float(metrics["relative"]["macro_balanced_accuracy"]),
                "relative_overall": float(metrics["relative"]["accuracy"]),
                "selected": layer == selected_layer,
            })

main_df = pd.DataFrame(main_rows).sort_values(["board", "architecture", "objective"])
layers_df = pd.DataFrame(layer_rows).sort_values(["board", "system", "layer"])
main_df.to_csv(OUTPUT_DIR / "chapter5_main_results.csv", index=False)

# Presentation-ready source for Table 3. The point estimates and percentile
# bootstrap limits are converted to percentages but remain numeric so that the
# thesis table can be regenerated without copying values by hand.
main_table = main_df[[
    "board", "system", "legal_readout",
    "top1_legal", "top1_ci_low", "top1_ci_high",
    "legal_probability_mass", "legal_mass_ci_low", "legal_mass_ci_high",
]].copy()
for column in (
    "top1_legal", "top1_ci_low", "top1_ci_high",
    "legal_probability_mass", "legal_mass_ci_low", "legal_mass_ci_high",
):
    main_table[column] = 100.0 * main_table[column]
main_table.to_csv(OUTPUT_DIR / "chapter5_main_results_table.csv", index=False)

layers_df.to_csv(OUTPUT_DIR / "chapter5_relative_probe_layers.csv", index=False)
(OUTPUT_DIR / "chapter5_main_results.json").write_text(
    json.dumps(compact_payload, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(main_df[[
    "system", "board", "legal_readout",
    "top1_legal", "top1_ci_low", "top1_ci_high",
    "legal_probability_mass", "legal_mass_ci_low", "legal_mass_ci_high",
    "linear_relative_macro", "linear_relative_overall",
]])
display(main_table.round(4))


,system,board,legal_readout,top1_legal,top1_ci_low,top1_ci_high,legal_probability_mass,legal_mass_ci_low,legal_mass_ci_high,linear_relative_macro,linear_relative_overall
6,Mamba-AR,8,native,0.998470,0.998411,0.998530,0.993445,0.993349,0.993541,0.978481,0.983235
9,Mamba-JEPA,8,mlp,0.997192,0.997104,0.997277,0.989041,0.988943,0.989145,0.980406,0.984782
0,Transformer-AR,8,native,0.997257,0.997168,0.997340,0.984772,0.984650,0.984897,0.969149,0.975942
3,Transformer-JEPA,8,linear,0.994156,0.994037,0.994281,0.977628,0.977518,0.977741,0.967244,0.974531
7,Mamba-AR,12,native,0.995118,0.995048,0.995187,0.954124,0.954027,0.954220,0.913673,0.934131
10,Mamba-JEPA,12,linear,0.989925,0.989813,0.990037,0.954723,0.954635,0.954807,0.928544,0.945419
1,Transformer-AR,12,native,0.994157,0.994084,0.994228,0.931353,0.931236,0.931474,0.891668,0.917353
4,Transformer-JEPA,12,linear,0.985240,0.985108,0.985370,0.933021,0.932918,0.933125,0.884753,0.912095
8,Mamba-AR,16,native,0.990493,0.990412,0.990581,0.907392,0.907285,0.907501,0.833498,0.873740
11,Mamba-JEPA,16,linear,0.984132,0.984026,0.984243,0.904246,0.904152,0.904335,0.837663,0.875616


,board,system,legal_readout,top1_legal,top1_ci_low,top1_ci_high,legal_probability_mass,legal_mass_ci_low,legal_mass_ci_high
6,8,Mamba-AR,native,99.8470,99.8411,99.8530,99.3445,99.3349,99.3541
9,8,Mamba-JEPA,mlp,99.7192,99.7104,99.7277,98.9041,98.8943,98.9145
0,8,Transformer-AR,native,99.7257,99.7168,99.7340,98.4772,98.4650,98.4897
3,8,Transformer-JEPA,linear,99.4156,99.4037,99.4281,97.7628,97.7518,97.7741
7,12,Mamba-AR,native,99.5118,99.5048,99.5187,95.4124,95.4027,95.4220
10,12,Mamba-JEPA,linear,98.9925,98.9813,99.0037,95.4723,95.4635,95.4807
1,12,Transformer-AR,native,99.4157,99.4084,99.4228,93.1353,93.1236,93.1474
4,12,Transformer-JEPA,linear,98.5240,98.5108,98.5370,93.3021,93.2918,93.3125
8,16,Mamba-AR,native,99.0493,99.0412,99.0581,90.7392,90.7285,90.7501
11,16,Mamba-JEPA,linear,98.4132,98.4026,98.4243,90.4246,90.4152,90.4335


In [3]:
def style_axis(ax):
    ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color=GRID, linewidth=0.8, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK2, length=0)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)

fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.7), dpi=220)
fig.patch.set_facecolor(SURFACE)
panels = [
    ("top1_legal", "Top-1 νόμιμη πρόβλεψη", "Ακρίβεια (%)", (97.2, 100.05)),
    ("linear_relative_macro", "Αποκωδικοποίηση σχετικής κατάστασης", "Macro-balanced accuracy (%)", (75, 100.2)),
]
for ax, (metric, title, ylabel, ylim) in zip(axes, panels):
    style_axis(ax)
    for system, _, _ in SYSTEMS:
        rows = main_df[main_df.system == system].sort_values("board")
        ax.plot(rows.board, 100 * rows[metric], color=COLORS[system], marker=MARKERS[system],
                linewidth=2.4, markersize=6.5, label=system, zorder=3)
    ax.set_title(title, fontsize=11.5, color=INK, fontweight="semibold", pad=12)
    ax.set_xlabel("Μέγεθος ταμπλό", color=INK2)
    ax.set_ylabel(ylabel, color=INK2)
    ax.set_xticks(BOARDS, [f"{b}×{b}" for b in BOARDS])
    ax.set_ylim(*ylim)
axes[0].legend(loc="lower left", frameon=False, fontsize=8.8)
fig.suptitle("Μεταβολή της λειτουργικής και αναπαραστασιακής επίδοσης με την κλίμακα",
             fontsize=13, fontweight="semibold", color=INK, y=1.02)
fig.tight_layout()
for path in (OUTPUT_DIR / "fig_ch5_scaling_profile.png", FIGURE_DIR / "fig_ch5_scaling_profile.png"):
    fig.savefig(path, facecolor=SURFACE, bbox_inches="tight")
plt.show()


<ipython-input-1-6b0885366a50>:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


<!-- CH5_LEGALITY_BREAKDOWN_SECTION -->
## 1.1 Top-k legality and normalized game phase

These tables and figures are derived directly from the same twelve canonical
seed-42 bf16 `unified_eval_v4` result files used above. AR systems use
`native_ar`; JEPA systems use the Linear or MLP frozen readout selected only
by validation legal-probability mass. The notebook reads `topk_legal` for
k = 1, 2, 3, 5 and `normalized_phase_bins` for the four quarters of each
game. Every exported long-form CSV retains the source result path,
checkpoint SHA-256, and split-manifest hash.


In [4]:
# CH5_LEGALITY_BREAKDOWN_SECTION
from matplotlib.colors import LinearSegmentedColormap

topk_rows, phase_rows = [], []
system_order = [system for system, _, _ in SYSTEMS]
phase_labels = ["0–25%", "25–50%", "50–75%", "75–100%"]

for _, row in main_df.iterrows():
    payload = load_json(Path(row["source_results"]))
    readout, legal = choose_legal_readout(payload, row["objective"])
    if readout != row["legal_readout"]:
        raise RuntimeError(
            f"Readout mismatch for {row['system']} {row['board']}×{row['board']}: "
            f"{readout} != {row['legal_readout']}"
        )

    topk = legal["topk_legal"]
    for k in (1, 2, 3, 5):
        topk_rows.append({
            "board": int(row["board"]),
            "system": row["system"],
            "architecture": row["architecture"],
            "objective": row["objective"],
            "legal_readout": readout,
            "k": k,
            "legal_precision_at_k": float(topk[str(k)]),
            "checkpoint_sha256": row["checkpoint_sha256"],
            "split_manifest_hash": row["split_manifest_hash"],
            "source_results": row["source_results"],
        })

    bins = legal["normalized_phase_bins"]
    if len(bins) != 4:
        raise RuntimeError(
            f"Expected four phase bins for {row['system']} {row['board']}×{row['board']}"
        )
    for phase in bins:
        phase_bin = int(phase["phase_bin"])
        phase_rows.append({
            "board": int(row["board"]),
            "system": row["system"],
            "architecture": row["architecture"],
            "objective": row["objective"],
            "legal_readout": readout,
            "phase_bin": phase_bin,
            "phase": phase_labels[phase_bin],
            "phase_start": float(phase["phase_start"]),
            "phase_end": float(phase["phase_end"]),
            "n_tokens": int(phase["n_tokens"]),
            "mean_legal_moves": float(phase["mean_legal_moves"]),
            "top1_legal": float(phase["top1_legal"]),
            "legal_probability_mass": float(phase["legal_probability_mass"]),
            "checkpoint_sha256": row["checkpoint_sha256"],
            "split_manifest_hash": row["split_manifest_hash"],
            "source_results": row["source_results"],
        })

topk_df = pd.DataFrame(topk_rows)
phase_df = pd.DataFrame(phase_rows)
topk_df["system"] = pd.Categorical(topk_df["system"], system_order, ordered=True)
phase_df["system"] = pd.Categorical(phase_df["system"], system_order, ordered=True)
topk_df = topk_df.sort_values(["board", "system", "k"]).reset_index(drop=True)
phase_df = phase_df.sort_values(["board", "system", "phase_bin"]).reset_index(drop=True)

topk_table = (
    topk_df.pivot(index=["board", "system", "legal_readout"],
                  columns="k", values="legal_precision_at_k")
    .reset_index()
    .rename(columns={1: "P@1", 2: "P@2", 3: "P@3", 5: "P@5"})
)
phase_table = (
    phase_df.pivot(index=["board", "system", "legal_readout"],
                   columns="phase", values="top1_legal")
    .reset_index()
)
phase_table = phase_table[
    ["board", "system", "legal_readout", *phase_labels]
]

topk_df.to_csv(OUTPUT_DIR / "chapter5_topk_legality.csv", index=False)
phase_df.to_csv(OUTPUT_DIR / "chapter5_legality_by_phase.csv", index=False)
topk_table.to_csv(OUTPUT_DIR / "chapter5_topk_legality_table.csv", index=False)
phase_table.to_csv(OUTPUT_DIR / "chapter5_legality_by_phase_table.csv", index=False)
(OUTPUT_DIR / "chapter5_legality_breakdown.json").write_text(
    json.dumps({
        "protocol": PROTOCOL,
        "selection": "AR native; JEPA validation-selected frozen readout",
        "topk": topk_rows,
        "normalized_phase_bins": phase_rows,
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

display_topk = topk_table.copy()
display_phase = phase_table.copy()
for column in ("P@1", "P@2", "P@3", "P@5"):
    display_topk[column] = 100 * display_topk[column]
for column in phase_labels:
    display_phase[column] = 100 * display_phase[column]
display(display_topk.round(2))
display(display_phase.round(2))

# Precision among the first k ranked predictions.
fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.6), dpi=220, sharey=True)
fig.patch.set_facecolor(SURFACE)
y_min = max(0.0, 100 * topk_df["legal_precision_at_k"].min() - 1.0)
for ax, board in zip(axes, BOARDS):
    style_axis(ax)
    for system in system_order:
        rows = topk_df[
            (topk_df["board"] == board) & (topk_df["system"] == system)
        ]
        ax.plot(
            rows["k"], 100 * rows["legal_precision_at_k"],
            color=COLORS[system], marker=MARKERS[system],
            linewidth=2.2, markersize=5.8, label=system, zorder=3,
        )
    ax.set_title(f"{board}×{board}", fontsize=11.5, fontweight="semibold", color=INK)
    ax.set_xlabel("k", color=INK2)
    ax.set_xticks([1, 2, 3, 5])
    ax.set_ylim(y_min, 100.15)
axes[0].set_ylabel("Νόμιμες κινήσεις στις πρώτες k προβλέψεις (%)", color=INK2)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False,
           bbox_to_anchor=(0.5, -0.01), fontsize=8.8)
fig.suptitle(
    "Νομιμότητα σε μεγαλύτερο βάθος της κατάταξης προβλέψεων",
    fontsize=13, fontweight="semibold", color=INK, y=1.02,
)
fig.tight_layout(rect=(0, 0.07, 1, 1))
for path in (
    OUTPUT_DIR / "fig_ch5_topk_legality.png",
    FIGURE_DIR / "fig_ch5_topk_legality.png",
):
    fig.savefig(path, facecolor=SURFACE, bbox_inches="tight")
plt.show()

# Top-1 legality in four normalized quarters of each game.
row_pairs = [(board, system) for board in BOARDS for system in system_order]
phase_matrix = np.array([
    [
        float(phase_df[
            (phase_df["board"] == board)
            & (phase_df["system"] == system)
            & (phase_df["phase_bin"] == phase_bin)
        ]["top1_legal"].iloc[0]) * 100
        for phase_bin in range(4)
    ]
    for board, system in row_pairs
])
heatmap_cmap = LinearSegmentedColormap.from_list(
    "thesis_teal", ["#f4f7f5", "#d9e9e2", "#9bc9b8", "#4f8f79"]
)
vmin = math.floor((phase_matrix.min() - 0.25) * 2) / 2
fig, ax = plt.subplots(figsize=(9.4, 7.3), dpi=220)
fig.patch.set_facecolor(SURFACE)
ax.set_facecolor(SURFACE)
image = ax.imshow(phase_matrix, aspect="auto", cmap=heatmap_cmap,
                  vmin=vmin, vmax=100.0)
ax.set_xticks(range(4), phase_labels)
ax.set_yticks(
    range(len(row_pairs)),
    [f"{board}×{board}   {system}" for board, system in row_pairs],
)
ax.set_xlabel("Κανονικοποιημένη φάση της παρτίδας", color=INK2, labelpad=10)
ax.tick_params(colors=INK2, length=0)
for side in ax.spines.values():
    side.set_visible(False)
for boundary in (3.5, 7.5):
    ax.axhline(boundary, color=SURFACE, linewidth=4)
threshold = (vmin + 100.0) / 2
for row_index in range(phase_matrix.shape[0]):
    for column_index in range(phase_matrix.shape[1]):
        value = phase_matrix[row_index, column_index]
        ax.text(
            column_index, row_index, f"{value:.2f}",
            ha="center", va="center", fontsize=8.2,
            color="white" if value >= threshold else INK,
        )
colorbar = fig.colorbar(image, ax=ax, fraction=0.032, pad=0.025)
colorbar.set_label("Top-1 νομιμότητα (%)", color=INK2)
colorbar.ax.tick_params(colors=INK2, length=0)
colorbar.outline.set_visible(False)
ax.set_title(
    "Top-1 νομιμότητα ανά φάση της παρτίδας",
    fontsize=13, fontweight="semibold", color=INK, pad=15,
)
fig.tight_layout()
for path in (
    OUTPUT_DIR / "fig_ch5_legality_by_phase.png",
    FIGURE_DIR / "fig_ch5_legality_by_phase.png",
):
    fig.savefig(path, facecolor=SURFACE, bbox_inches="tight")
plt.show()


<ipython-input-1-41c960499fde>:136: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
<ipython-input-1-41c960499fde>:194: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


k,board,system,legal_readout,P@1,P@2,P@3,P@5
0,8,Transformer-AR,native,99.72,98.28,96.78,92.45
1,8,Transformer-JEPA,linear,99.42,98.01,96.55,92.26
2,8,Mamba-AR,native,99.84,98.39,96.90,92.57
3,8,Mamba-JEPA,mlp,99.72,98.29,96.80,92.49
4,12,Transformer-AR,native,99.42,98.76,98.12,96.34
5,12,Transformer-JEPA,linear,98.52,97.89,97.27,95.54
6,12,Mamba-AR,native,99.51,98.89,98.27,96.53
7,12,Mamba-JEPA,linear,98.99,98.39,97.81,96.13
8,16,Transformer-AR,native,98.99,98.55,98.12,97.01
9,16,Transformer-JEPA,linear,97.54,96.99,96.49,95.30


phase,board,system,legal_readout,0–25%,25–50%,50–75%,75–100%
0,8,Transformer-AR,native,99.99,99.86,99.66,99.40
1,8,Transformer-JEPA,linear,99.99,99.71,99.20,98.78
2,8,Mamba-AR,native,100.00,99.96,99.81,99.63
3,8,Mamba-JEPA,mlp,100.00,99.93,99.66,99.30
4,12,Transformer-AR,native,99.71,99.32,99.31,99.32
5,12,Transformer-JEPA,linear,99.46,98.46,98.21,97.97
6,12,Mamba-AR,native,99.89,99.45,99.39,99.32
7,12,Mamba-JEPA,linear,99.79,98.91,98.73,98.54
8,16,Transformer-AR,native,99.28,98.77,98.90,99.00
9,16,Transformer-JEPA,linear,98.33,96.99,97.33,97.49


In [5]:
fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.3), dpi=220, sharey=True)
fig.patch.set_facecolor(SURFACE)
for ax, board in zip(axes, BOARDS):
    style_axis(ax)
    for system, _, _ in SYSTEMS:
        rows = layers_df[(layers_df.board == board) & (layers_df.system == system)]
        ax.plot(rows.normalized_depth, 100 * rows.relative_macro, color=COLORS[system],
                linewidth=2.1, marker=MARKERS[system], markersize=3.8, label=system)
        chosen = rows[rows.selected]
        ax.scatter(chosen.normalized_depth, 100 * chosen.relative_macro, s=72,
                   facecolor=SURFACE, edgecolor=COLORS[system], linewidth=2.1, zorder=5)
    ax.set_title(f"{board}×{board}", fontsize=11.5, fontweight="semibold", color=INK)
    ax.set_xlabel("Κανονικοποιημένο βάθος", color=INK2)
    ax.set_xticks([0, .25, .5, .75, 1], ["Emb.", "25%", "50%", "75%", "Τελικό"])
    ax.set_ylim(55, 100.5)
axes[0].set_ylabel("Relative linear probe — macro-balanced accuracy (%)", color=INK2)
axes[0].legend(loc="lower right", frameon=False, fontsize=8.2)
fig.suptitle("Εξέλιξη της αποκωδικοποίησης της σχετικής κατάστασης ανά βάθος",
             fontsize=13, fontweight="semibold", color=INK, y=1.02)
fig.tight_layout()
for path in (OUTPUT_DIR / "fig_ch5_relative_probe_layers.png", FIGURE_DIR / "fig_ch5_relative_probe_layers.png"):
    fig.savefig(path, facecolor=SURFACE, bbox_inches="tight")
plt.show()


<ipython-input-1-bc8a86f7bfd2>:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Five-seed analysis at 8×8 and 16×16

The canonical seed 42 is combined with the four additional replications
0–3. The cell refuses an incomplete or unbalanced grid. Confidence intervals
use Student's t distribution over independent trainings; paired effects use
the seed as a block. Exact sign-flip p-values and Holm-adjusted paired-t
p-values are exported for audit.


In [6]:
SEEDS = (0, 1, 2, 3, 42)
SEED_BOARDS = (8, 16)

# The four additional seeds have already been validated by the seeded Common
# Evaluation and exported as compact raw-per-seed tables. Reusing those tables
# avoids repeatedly loading more than a gigabyte of full per-position JSON.
# Seed 42 is read from the freshly validated canonical grid above.
frames = []
for board in SEED_BOARDS:
    path = RESULTS_ROOT / "seeded_results" / "aggregate" / f"{board}x{board}" / f"raw_per_seed_b{board}.csv"
    if not path.is_file():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path)
    frame = frame[frame["seed"].isin((0, 1, 2, 3))].copy()
    if set(frame["seed"]) != {0, 1, 2, 3} or len(frame) != 16:
        raise RuntimeError(f"Expected a balanced four-seed grid in {path}")
    if set(frame["training_precision"].str.lower()) != {"bf16"}:
        raise RuntimeError(f"Non-bf16 replication found in {path}")
    frame = frame.rename(columns={
        "board_size": "board",
        "legal_head": "legal_readout",
        "legal_preferred_top1": "top1_legal",
        "legal_preferred_mass": "legal_probability_mass",
        "board_linear_relative": "linear_relative_macro",
        "results_path": "source_results",
    })
    # The compact historical export reports the macro metric used for layer
    # selection. Overall relative accuracy was not part of that export and is
    # therefore left missing for seeds 0–3 rather than reconstructed.
    frame["linear_relative_overall"] = np.nan
    frames.append(frame[[
        "system", "architecture", "objective", "board", "seed",
        "legal_readout", "top1_legal", "legal_probability_mass",
        "linear_relative_macro", "linear_relative_overall",
        "checkpoint_sha256", "source_results",
    ]])

seed42 = main_df[main_df.board.isin(SEED_BOARDS)][[
    "system", "architecture", "objective", "board", "seed",
    "legal_readout", "top1_legal", "legal_probability_mass",
    "linear_relative_macro", "linear_relative_overall",
    "checkpoint_sha256", "source_results",
]].copy()
if set(seed42.seed) != {42} or len(seed42) != 8:
    raise RuntimeError("Canonical grid does not contain the eight expected seed-42 cells")

seed_df = pd.concat([*frames, seed42], ignore_index=True).sort_values(["board", "system", "seed"])
expected = len(SYSTEMS) * len(SEED_BOARDS) * len(SEEDS)
if len(seed_df) != expected:
    raise RuntimeError(f"Expected {expected} seeded cells, found {len(seed_df)}")
for (board, system), group in seed_df.groupby(["board", "system"]):
    if set(group.seed) != set(SEEDS):
        raise RuntimeError(f"Unbalanced seeds for {system} {board}x{board}: {sorted(group.seed)}")
seed_df.to_csv(OUTPUT_DIR / "chapter5_seeded_raw.csv", index=False)

def t_interval(values):
    values = np.asarray(values, dtype=float)
    mean, sd, n = values.mean(), values.std(ddof=1), len(values)
    half = stats.t.ppf(.975, n - 1) * sd / math.sqrt(n)
    return mean, sd, mean - half, mean + half

summary_rows = []
for (board, system), group in seed_df.groupby(["board", "system"], sort=True):
    for metric in ("top1_legal", "legal_probability_mass", "linear_relative_macro"):
        mean, sd, low, high = t_interval(group[metric])
        summary_rows.append({"board": board, "system": system, "metric": metric,
                             "n": len(group), "mean": mean, "sd": sd,
                             "ci95_t_low": low, "ci95_t_high": high})
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "chapter5_seeded_summary.csv", index=False)

def exact_sign_flip_p(values):
    values = np.asarray(values, dtype=float)
    observed = abs(values.mean())
    permuted = [abs((values * np.asarray(signs)).mean())
                for signs in product((-1.0, 1.0), repeat=len(values))]
    return sum(value >= observed - 1e-15 for value in permuted) / len(permuted)

def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    order = np.argsort(pvalues)
    adjusted = np.empty_like(pvalues)
    running = 0.0
    m = len(pvalues)
    for rank, index in enumerate(order):
        running = max(running, (m - rank) * pvalues[index])
        adjusted[index] = min(1.0, running)
    return adjusted

effect_rows = []
metrics = ("top1_legal", "legal_probability_mass", "linear_relative_macro")
for board in SEED_BOARDS:
    block = seed_df[seed_df.board == board]
    for metric in metrics:
        pivot = block.pivot(index="seed", columns=["architecture", "objective"], values=metric).sort_index()
        contrasts = {
            "architecture_main": ((pivot[("mamba", "ar")] + pivot[("mamba", "jepa")]) / 2 -
                                  (pivot[("transformer", "ar")] + pivot[("transformer", "jepa")]) / 2),
            "objective_main": ((pivot[("transformer", "jepa")] + pivot[("mamba", "jepa")]) / 2 -
                               (pivot[("transformer", "ar")] + pivot[("mamba", "ar")]) / 2),
            "interaction": ((pivot[("mamba", "jepa")] - pivot[("mamba", "ar")]) -
                            (pivot[("transformer", "jepa")] - pivot[("transformer", "ar")])),
        }
        family_start = len(effect_rows)
        for effect, values in contrasts.items():
            mean, sd, low, high = t_interval(values)
            t_result = stats.ttest_1samp(values, 0.0)
            correction = 1 - 3 / (4 * (len(values) - 1) - 1)
            gz = correction * mean / sd if sd > 0 else math.nan
            effect_rows.append({"board": board, "metric": metric, "effect": effect, "n": len(values),
                                "mean_difference": mean, "sd_difference": sd,
                                "ci95_low": low, "ci95_high": high,
                                "t_statistic": float(t_result.statistic), "p_paired_t": float(t_result.pvalue),
                                "p_exact_sign_flip": exact_sign_flip_p(values), "hedges_gz": gz,
                                "wins_positive": int((values > 0).sum()), "ties": int((values == 0).sum()),
                                "wins_negative": int((values < 0).sum())})
        family = effect_rows[family_start:family_start + 3]
        adjusted = holm_adjust([row["p_paired_t"] for row in family])
        for row, value in zip(family, adjusted):
            row["p_holm"] = value
            row["significant_holm_0_05"] = bool(value < .05)

effects_df = pd.DataFrame(effect_rows)
effects_df.to_csv(OUTPUT_DIR / "chapter5_seeded_factorial_effects.csv", index=False)

# Presentation-ready sources for Appendix F. These retain numeric values and
# use the same ordering and sign conventions as the main factorial table.
metric_labels = {
    "top1_legal": "Top-1 νομιμότητα",
    "legal_probability_mass": "Νόμιμη μάζα",
    "linear_relative_macro": "Σχετική κατάσταση",
}
effect_labels = {
    "architecture_main": "Αρχιτεκτονική",
    "objective_main": "Objective",
    "interaction": "Αλληλεπίδραση",
}
appendix_effects = effects_df.copy()
appendix_effects["metric_label"] = appendix_effects["metric"].map(metric_labels)
appendix_effects["effect_label"] = appendix_effects["effect"].map(effect_labels)
for column in ("mean_difference", "sd_difference", "ci95_low", "ci95_high"):
    appendix_effects[f"{column}_pp"] = 100.0 * appendix_effects[column]

appendix_parametric = appendix_effects[[
    "board", "metric_label", "effect_label",
    "mean_difference_pp", "ci95_low_pp", "ci95_high_pp",
    "p_paired_t", "p_holm", "significant_holm_0_05",
]].copy()
appendix_robustness = appendix_effects[[
    "board", "metric_label", "effect_label",
    "p_exact_sign_flip", "wins_positive", "ties", "wins_negative", "hedges_gz",
]].copy()
appendix_parametric.to_csv(
    OUTPUT_DIR / "chapter5_appendix_parametric_tests.csv", index=False
)
appendix_robustness.to_csv(
    OUTPUT_DIR / "chapter5_appendix_signflip_hedges.csv", index=False
)
(OUTPUT_DIR / "chapter5_seeded_results.json").write_text(
    json.dumps({"protocol": PROTOCOL, "seeds": list(SEEDS),
                "raw": json.loads(seed_df.to_json(orient="records")),
                "summaries": summary_rows, "factorial_effects": effect_rows},
               ensure_ascii=False, indent=2,
               default=lambda value: value.item() if isinstance(value, np.generic) else str(value)), encoding="utf-8"
)
display(summary_df)
display(effects_df)


,board,system,metric,n,mean,sd,ci95_t_low,ci95_t_high
0,8,Mamba-AR,top1_legal,5,0.998376,0.000137,0.998206,0.998546
1,8,Mamba-AR,legal_probability_mass,5,0.993746,0.000546,0.993069,0.994424
2,8,Mamba-AR,linear_relative_macro,5,0.979169,0.001409,0.977419,0.980918
3,8,Mamba-JEPA,top1_legal,5,0.997240,0.000044,0.997186,0.997295
4,8,Mamba-JEPA,legal_probability_mass,5,0.989754,0.000461,0.989182,0.990327
5,8,Mamba-JEPA,linear_relative_macro,5,0.981747,0.000762,0.980802,0.982693
6,8,Transformer-AR,top1_legal,5,0.997126,0.000115,0.996984,0.997268
7,8,Transformer-AR,legal_probability_mass,5,0.984437,0.000605,0.983685,0.985188
8,8,Transformer-AR,linear_relative_macro,5,0.969527,0.000660,0.968707,0.970347
9,8,Transformer-JEPA,top1_legal,5,0.994193,0.000135,0.994025,0.994361


,board,metric,effect,n,mean_difference,sd_difference,ci95_low,ci95_high,t_statistic,p_paired_t,p_exact_sign_flip,hedges_gz,wins_positive,ties,wins_negative,p_holm,significant_holm_0_05
0,8,top1_legal,architecture_main,5,0.002149,0.000038,0.002102,0.002196,125.978847,2.381104e-08,0.0625,45.071562,5,0,0,7.143311e-08,True
1,8,top1_legal,objective_main,5,-0.002034,0.000113,-0.002174,-0.001894,-40.323809,2.260094e-06,0.0625,-14.426684,0,0,5,4.520189e-06,True
2,8,top1_legal,interaction,5,0.001797,0.000217,0.001527,0.002067,18.490625,5.034123e-05,0.0625,6.615407,5,0,0,5.034123e-05,True
3,8,legal_probability_mass,architecture_main,5,0.010714,0.000945,0.009541,0.011887,25.362330,1.435182e-05,0.0625,9.073903,5,0,0,3.283986e-05,True
4,8,legal_probability_mass,objective_main,5,-0.005396,0.000444,-0.005948,-0.004844,-27.148025,1.094662e-05,0.0625,-9.712773,0,0,5,3.283986e-05,True
5,8,legal_probability_mass,interaction,5,0.002808,0.000526,0.002155,0.003462,11.930152,2.828095e-04,0.0625,4.268261,5,0,0,2.828095e-04,True
6,8,linear_relative_macro,architecture_main,5,0.012614,0.001280,0.011025,0.014204,22.036610,2.509767e-05,0.0625,7.884057,5,0,0,7.529301e-05,True
7,8,linear_relative_macro,objective_main,5,-0.000394,0.001117,-0.001780,0.000992,-0.789037,4.742273e-01,0.5000,-0.282294,2,0,3,4.742273e-01,False
8,8,linear_relative_macro,interaction,5,0.005945,0.003625,0.001444,0.010446,3.667328,2.144149e-02,0.0625,1.312063,5,0,0,4.288299e-02,True
9,16,top1_legal,architecture_main,5,0.004793,0.000699,0.003924,0.005661,15.322851,1.058184e-04,0.0625,5.482070,5,0,0,2.116367e-04,True


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.8), dpi=220)
fig.patch.set_facecolor(SURFACE)
for ax, (metric, title, ylabel, ylim) in zip(axes, [
    ("top1_legal", "Top-1 νόμιμη πρόβλεψη", "Ακρίβεια (%)", (97.0, 100.05)),
    ("linear_relative_macro", "Σχετική κατάσταση — linear probe", "Macro-balanced accuracy (%)", (74, 100.2)),
]):
    style_axis(ax)
    xbase = np.arange(len(SEED_BOARDS))
    offsets = np.linspace(-.24, .24, len(SYSTEMS))
    for offset, (system, _, _) in zip(offsets, SYSTEMS):
        group = seed_df[seed_df.system == system]
        for x, board in zip(xbase, SEED_BOARDS):
            values = 100 * group[group.board == board].sort_values("seed")[metric].to_numpy()
            jitter = np.linspace(-.025, .025, len(values))
            ax.scatter(np.full(len(values), x + offset) + jitter, values, s=20,
                       color=COLORS[system], alpha=.75, zorder=3)
            mean, _, low, high = t_interval(values)
            ax.errorbar(x + offset, mean, yerr=[[mean-low], [high-mean]], fmt=MARKERS[system],
                        color=COLORS[system], markeredgecolor=INK, markeredgewidth=.5,
                        markersize=6.5, linewidth=1.5, capsize=3, label=system if x == 0 else None, zorder=4)
    ax.set_xticks(xbase, ["8×8", "16×16"])
    ax.set_ylim(*ylim)
    ax.set_title(title, fontsize=11.5, fontweight="semibold", color=INK)
    ax.set_ylabel(ylabel, color=INK2)
axes[0].legend(loc="lower left", frameon=False, fontsize=8.4)
fig.suptitle("Μεταβλητότητα μεταξύ πέντε ανεξάρτητων εκπαιδεύσεων",
             fontsize=13, fontweight="semibold", color=INK, y=1.02)
fig.tight_layout()
for path in (OUTPUT_DIR / "fig_ch5_five_seed_results.png", FIGURE_DIR / "fig_ch5_five_seed_results.png"):
    fig.savefig(path, facecolor=SURFACE, bbox_inches="tight")
plt.show()


<ipython-input-1-c028cd7b45d1>:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Causal-suite completeness audit and figure

The required final artifact is `full/readout_native` for AR and
`full/readout_best` for JEPA. Older root-level `full` outputs are not silently
accepted for JEPA because they used a fixed MLP readout. A missing cell is
listed and the final causal figure is withheld.


In [ ]:
CAUSAL_SUITE_ID = "causal_intervention_suite_v2"
ADAPTED_PROTOCOL_ID = "adapted_linear_sequential_residual_v2"
ADAPTED_RANDOM_CONTROL_ID = "shared_signed_permutation_v1"


METHOD_VARIANTS = {
    "li": "optimized_activation",
    "nanda": "target_class_direction",
    "adapted": "probe_direction",
}

causal_rows, missing_causal = [], []
for system, architecture, objective in SYSTEMS:
    for board in BOARDS:
        run_dir = ARTIFACTS_ROOT / "runs" / RUNS[(system, board)]
        readout_run = "native" if objective == "ar" else "best"
        path = (run_dir / "causal_intervention" / CAUSAL_SUITE_ID /
                "full" / f"readout_{readout_run}" / f"results__{CAUSAL_SUITE_ID}.json")
        if not path.is_file():
            missing_causal.append(f"{system} {board}×{board}: {path}")
            continue
        state = load_json(path)
        expected_hash = load_json(main_result_path(system, board))["metadata"]["checkpoint_sha256"]
        if state["identity"].get("checkpoint_sha256") != expected_hash:
            missing_causal.append(f"{system} {board}×{board}: checkpoint mismatch in {path}")
            continue
        if objective == "jepa":
            selection = state["identity"].get("readout_selection") or {}
            if selection.get("requested_policy") != "best":
                missing_causal.append(f"{system} {board}×{board}: no validation-selected best readout in {path}")
                continue
        for method, variant in METHOD_VARIANTS.items():
            result = state.get("methods", {}).get(method)
            if result is None:
                missing_causal.append(f"{system} {board}×{board}: method {method} incomplete")
                continue
            if method == "adapted" and (
                result.get("protocol") != ADAPTED_PROTOCOL_ID
                or state["identity"].get("adapted_random_control") != ADAPTED_RANDOM_CONTROL_ID
            ):
                missing_causal.append(
                    f"{system} {board}×{board}: adapted random control requires refresh in {path}"
                )
                continue
            baseline = result["test"]["null"]
            edited = result["test"][variant]
            random_control = result["test"].get("geometry_matched_random")
            causal_rows.append({
                "system": system, "board": board, "method": method,
                "readout": (state["identity"].get("readout_selection") or {}).get("selected", "native_ar"),
                "n_cases": int(edited["n_cases"]),
                "target_top1_legal": float(edited["target_top1_legal"]),
                "delta_top1_pp": 100 * (float(edited["target_top1_legal"]) - float(baseline["target_top1_legal"])),
                "target_legal_mass": float(edited["target_legal_probability_mass"]),
                "delta_legal_mass_pp": 100 * (float(edited["target_legal_probability_mass"]) - float(baseline["target_legal_probability_mass"])),
                "topn_error": float(edited["mean_topn_false_positive_plus_false_negative"]),
                "delta_topn_error": float(edited["mean_topn_false_positive_plus_false_negative"]) - float(baseline["mean_topn_false_positive_plus_false_negative"]),
                "delta_vs_random_top1_pp": (100 * (float(edited["target_top1_legal"]) - float(random_control["target_top1_legal"])) if random_control else None),
                "delta_vs_random_legal_mass_pp": (100 * (float(edited["target_legal_probability_mass"]) - float(random_control["target_legal_probability_mass"])) if random_control else None),
                "delta_vs_random_topn_error": (float(edited["mean_topn_false_positive_plus_false_negative"]) - float(random_control["mean_topn_false_positive_plus_false_negative"]) if random_control else None),
                "source_results": str(path),
            })

if missing_causal:
    print("CAUSAL FIGURE WITHHELD — complete or refresh these cells:")
    for item in missing_causal:
        print("-", item)
else:
    causal_df = pd.DataFrame(causal_rows)
    causal_df.to_csv(OUTPUT_DIR / "chapter5_causal_results.csv", index=False)
    (OUTPUT_DIR / "chapter5_causal_results.json").write_text(
        json.dumps(causal_rows, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    method_labels = {"li": "Li", "nanda": "Nanda", "adapted": "Προσαρμοσμένη"}
    fig, axes = plt.subplots(1, 3, figsize=(14.2, 5.1), dpi=220, sharey=True)
    fig.patch.set_facecolor(SURFACE)
    labels = [system for system, _, _ in SYSTEMS]
    y = np.arange(len(labels))
    for ax, board in zip(axes, BOARDS):
        style_axis(ax)
        block = causal_df[causal_df.board == board]
        offsets = {"li": -.18, "nanda": 0, "adapted": .18}
        for method in METHOD_VARIANTS:
            values = [float(block[(block.system == system) & (block.method == method)].delta_legal_mass_pp.iloc[0]) for system in labels]
            ax.scatter(values, y + offsets[method], s=48, label=method_labels[method],
                       color={"li": "#6F9FC8", "nanda": "#D6A24B", "adapted": "#79B7A3"}[method], zorder=3)
        ax.axvline(0, color=INK2, linewidth=.9)
        ax.set_title(f"{board}×{board}", fontsize=11.5, fontweight="semibold")
        ax.set_xlabel("Μεταβολή νόμιμης μάζας (ποσοστιαίες μονάδες)", color=INK2)
        ax.grid(axis="x", color=GRID, linewidth=.8)
        ax.grid(axis="y", visible=False)
        ax.set_yticks(y)
        if ax is axes[0]:
            ax.set_yticklabels(labels)
        else:
            ax.tick_params(axis="y", labelleft=False)
    axes[0].invert_yaxis()
    handles, legend_labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, legend_labels, loc="lower center", ncol=3, frameon=False, fontsize=9)
    fig.suptitle("Αιτιακή μεταβολή προς το αντιπαραθετικό σύνολο νόμιμων κινήσεων",
                 fontsize=13, fontweight="semibold", y=1.02)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    for out in (OUTPUT_DIR / "fig_ch5_causal_effects.png", FIGURE_DIR / "fig_ch5_causal_effects.png"):
        fig.savefig(out, facecolor=SURFACE, bbox_inches="tight")
    plt.show()


## 4. Supplementary probe and causal results

Validation-selected absolute/relative linear/MLP probes, class recalls, final-layer metrics, normalized lift and all causal outcome measures. All values come from existing result artifacts; no models are rerun.

In [9]:
# SUPPLEMENTARY_RESULT_TABLES_V1
# Reporting only: reuse saved test metrics and validation-selected layers.
import hashlib

probe_rows, baseline_rows, provenance_rows = [], [], []
for system, architecture, objective in SYSTEMS:
    for board in BOARDS:
        path = main_result_path(system, board)
        payload = load_json(path)
        meta = payload['metadata']
        assert int(meta['board_size']) == board
        assert str(meta.get('training_precision') or meta['train_config']['precision']).lower() == 'bf16'
        assert int(meta['train_config']['seed']) == 42
        file_hash = hashlib.sha256(path.read_bytes()).hexdigest()
        provenance_rows.append(dict(system=system, board=board, source_results=str(path), sha256=file_hash))
        for family in ('linear', 'mlp'):
            bank = payload['board_state'][family]
            assert bank['input_layernorm'] is True, (system, board, family)
            max_layer = max(map(int, bank['test_metrics']))
            for encoding in ('absolute', 'relative'):
                selected = bank['selected'][encoding]
                assert selected['selection_metric'] == 'macro_balanced_accuracy'
                test, final = selected['test'], bank['final_layer_test'][encoding]
                recall = test['per_class_recall']
                row = dict(system=system, board=board, probe=family, encoding=encoding,
                    input_layernorm=bank['input_layernorm'], layer=int(selected['layer']),
                    max_layer=max_layer, normalized_depth=int(selected['layer']) / max_layer,
                    overall=100 * float(test['accuracy']), macro=100 * float(test['macro_balanced_accuracy']),
                    empty=100 * float(test['empty_accuracy']), occupied=100 * float(test['occupied_accuracy']),
                    final_overall=100 * float(final['accuracy']), final_macro=100 * float(final['macro_balanced_accuracy']),
                    n_labels=int(test['n_labels']), n_occupied_labels=int(test['n_occupied_labels']),
                    source_results=str(path), source_sha256=file_hash)
                row.update({f'recall_{name}': 100 * float(value) for name,value in recall.items()})
                assert abs(row['macro'] - sum(recall.values()) / 3 * 100) < 1e-8
                assert abs(row['overall'] - (row['occupied'] * test['n_occupied_labels'] + row['empty'] * test['n_empty_labels']) / test['n_labels']) < 1e-8
                probe_rows.append(row)
        readout, legal = choose_legal_readout(payload, objective)
        baseline = float(legal['random_vocab_top1_legal_baseline'])
        top1 = float(legal['top1_legal_per_token'])
        lift = float(legal['top1_legal_normalized_lift'])
        assert abs(lift - (top1 - baseline) / (1 - baseline)) < 1e-8
        baseline_rows.append(dict(system=system, board=board, readout=readout,
            baseline_percent=100 * baseline, top1_per_token_percent=100 * top1,
            normalized_lift_percent=100 * lift, source_results=str(path)))

supplement_probes = pd.DataFrame(probe_rows).sort_values(['board', 'system', 'probe', 'encoding'])
supplement_baselines = pd.DataFrame(baseline_rows).sort_values(['board', 'system'])
assert len(supplement_probes) == 48 and len(supplement_baselines) == 12
supplement_probes.to_csv(OUTPUT_DIR / 'chapter5_probe_comparison_detailed.csv', index=False)
supplement_baselines.to_csv(OUTPUT_DIR / 'chapter5_normalized_lift.csv', index=False)

supplement_causal = pd.read_csv(OUTPUT_DIR / 'chapter5_causal_results.csv')
assert len(supplement_causal) == 36
assert set(supplement_causal['n_cases']) == {1000}
assert supplement_causal.groupby(['system', 'board']).size().eq(3).all()
causal_tables = {}
for name, delta, random_delta in (
    ('top1', 'delta_top1_pp', 'delta_vs_random_top1_pp'),
    ('topn_error', 'delta_topn_error', 'delta_vs_random_topn_error'),
):
    table = supplement_causal.pivot(index=['board', 'system'], columns='method', values=delta)
    table = table[['li', 'nanda', 'adapted']]
    table['adapted_minus_random'] = supplement_causal[supplement_causal.method.eq('adapted')].set_index(['board', 'system'])[random_delta]
    table = table.reset_index()
    assert table.notna().all().all()
    table.to_csv(OUTPUT_DIR / f'chapter5_causal_{name}_table.csv', index=False)
    causal_tables[name] = json.loads(table.to_json(orient='records'))

supplement = dict(protocol=PROTOCOL, canonical_seed=42,
    probe_selection='validation macro_balanced_accuracy; no selection on test',
    probe_values='percent; both families use input LayerNorm',
    normalized_lift='(top1_per_token - random_vocab_baseline) / (1 - random_vocab_baseline)',
    uncertainty='No new bootstrap intervals: probe and causal artifacts have no such intervals.',
    probes=json.loads(supplement_probes.to_json(orient='records')),
    baselines=json.loads(supplement_baselines.to_json(orient='records')),
    causal_tables=causal_tables, provenance=provenance_rows)
(OUTPUT_DIR / 'chapter5_supplementary_results.json').write_text(json.dumps(supplement, ensure_ascii=False, indent=2), encoding='utf-8')
display(supplement_probes[['board','system','probe','encoding','layer','overall','macro']].round(3))
display(supplement_baselines.drop(columns='source_results').round(3))
for name in causal_tables:
    display(pd.DataFrame(causal_tables[name]).round(3))
print('Supplementary tables regenerated from all twelve canonical result files; no training or evaluation rerun.')


Supplementary tables regenerated from all twelve canonical result files; no training or evaluation rerun.


,board,system,probe,encoding,layer,overall,macro
24,8,Mamba-AR,linear,absolute,11,75.498,68.782
25,8,Mamba-AR,linear,relative,12,98.324,97.848
26,8,Mamba-AR,mlp,absolute,10,89.434,86.551
27,8,Mamba-AR,mlp,relative,12,98.160,97.618
36,8,Mamba-JEPA,linear,absolute,12,75.602,68.919
37,8,Mamba-JEPA,linear,relative,13,98.478,98.041
38,8,Mamba-JEPA,mlp,absolute,9,91.412,89.306
39,8,Mamba-JEPA,mlp,relative,12,98.302,97.816
0,8,Transformer-AR,linear,absolute,3,75.573,68.878
1,8,Transformer-AR,linear,relative,6,97.594,96.915


,system,board,readout,baseline_percent,top1_per_token_percent,normalized_lift_percent
6,Mamba-AR,8,native,14.126,99.847,99.822
9,Mamba-JEPA,8,mlp,14.126,99.719,99.673
0,Transformer-AR,8,native,14.126,99.726,99.681
3,Transformer-JEPA,8,linear,14.126,99.416,99.319
7,Mamba-AR,12,native,11.856,99.511,99.446
10,Mamba-JEPA,12,linear,11.856,98.992,98.856
1,Transformer-AR,12,native,11.856,99.415,99.337
4,Transformer-JEPA,12,linear,11.856,98.523,98.324
8,Mamba-AR,16,native,10.289,99.049,98.940
11,Mamba-JEPA,16,linear,10.289,98.412,98.230


,board,system,li,nanda,adapted,adapted_minus_random
0,8,Mamba-AR,0.7,0.2,6.4,7.1
1,8,Mamba-JEPA,1.1,0.1,7.9,6.8
2,8,Transformer-AR,5.4,1.2,6.7,7.6
3,8,Transformer-JEPA,4.8,1.4,6.5,7.7
4,12,Mamba-AR,0.9,0.4,3.5,3.8
5,12,Mamba-JEPA,1.6,0.2,4.7,3.8
6,12,Transformer-AR,-1.0,0.9,2.7,4.4
7,12,Transformer-JEPA,0.2,1.2,3.2,2.6
8,16,Mamba-AR,0.4,0.4,2.6,3.0
9,16,Mamba-JEPA,-0.3,0.1,1.3,0.0


,board,system,li,nanda,adapted,adapted_minus_random
0,8,Mamba-AR,-0.140,-0.024,-2.034,-2.040
1,8,Mamba-JEPA,-0.070,-0.010,-2.168,-2.128
2,8,Transformer-AR,-0.934,-0.508,-1.862,-1.886
3,8,Transformer-JEPA,-0.688,-0.324,-2.116,-2.124
4,12,Mamba-AR,0.022,-0.024,-1.498,-1.602
5,12,Mamba-JEPA,0.010,-0.018,-1.628,-1.634
6,12,Transformer-AR,2.666,-0.628,-1.240,-1.414
7,12,Transformer-JEPA,3.136,-0.692,-1.248,-1.286
8,16,Mamba-AR,0.164,-0.008,-0.814,-0.944
9,16,Mamba-JEPA,0.154,-0.038,-0.804,-0.978
